# Chapter 44 — Responsible AI: Fairness, Governance, and Law

*From Absolute Zero* — companion notebook.

Every block below is the code printed in the chapter, in the same order. Run the cells top to bottom; the output should match the book exactly. If it does not, check `requirements.txt` first, then `docs/TROUBLESHOOTING.md`.

In [1]:
!pip -q install -r https://raw.githubusercontent.com/FromAbsoluteZero/CodeBase/main/requirements.txt  # Colab only; skip locally

zsh:1: command not found: pip


## Shared setup

Imports and the objects the blocks below reuse. The chapter prints these once and then continues the same session. This cell is `code/ch44/_lib.py`.

In [2]:
import numpy as np, warnings; warnings.filterwarnings("ignore")
from sklearn.linear_model import LogisticRegression

## The chapter code

### Block 1  (`c1.py`)

In [3]:
# A synthetic lending scenario, built so the true qualification rate
# genuinely differs between two groups -- exactly the condition under
# which fairness definitions are known to conflict. Group membership
# itself is not a model input; only correlated features are, and each
# individual's features are drawn conditional on their OWN true
# repayment outcome, so the model has genuine signal to learn from.
r = np.random.default_rng(44)
n = 4000
group = r.integers(0, 2, n)
# true qualification rate per group
base_rate = np.where(group == 0, 0.75, 0.55)
will_repay = r.random(n) < base_rate
y = will_repay.astype(int)

# features genuinely separate repayers from defaulters, with a
# group-level offset layered on top, reflecting real disparities in
# income and credit history that correlate with, but do not equal,
# group membership
income = r.normal(45 + 20 * y + 8 * (group == 0), 10, n)
credit_history = r.normal(50 + 25 * y + 5 * (group == 0), 12, n)

X = np.column_stack([income, credit_history])
model = LogisticRegression().fit(X, y)
p_repay = model.predict_proba(X)[:, 1]
pred = (p_repay >= 0.5).astype(int)

print(f"n = {n}, group 0: {np.sum(group==0)}, group 1: {np.sum(group==1)}")
print(f"true qualification rate, group 0: {y[group==0].mean():.4f}")
print(f"true qualification rate, group 1: {y[group==1].mean():.4f}")
print(f"overall model accuracy: {(pred == y).mean():.4f}")
print(f"\napproval rate (model), group 0: {pred[group==0].mean():.4f}")
print(f"approval rate (model), group 1: {pred[group==1].mean():.4f}")
print(f"demographic parity gap: "
      f"{abs(pred[group==0].mean() - pred[group==1].mean()):.4f}")

n = 4000, group 0: 2007, group 1: 1993
true qualification rate, group 0: 0.7534
true qualification rate, group 1: 0.5529
overall model accuracy: 0.9263

approval rate (model), group 0: 0.7932
approval rate (model), group 1: 0.5193
demographic parity gap: 0.2739


### Block 2  (`c2.py`)

In [4]:
# Demographic parity is only one definition of fairness. Equalized odds
# asks a different question: among people who would actually repay, is
# the model equally likely to approve them, regardless of group? And
# among people who would default, is it equally likely to correctly
# reject them?
def group_rates(y_true, y_pred, group_mask):
    y_t, y_p = y_true[group_mask], y_pred[group_mask]
    # true positive rate: approved among true repayers
    tpr = y_p[y_t == 1].mean()
    # false positive rate: approved among true defaulters
    fpr = y_p[y_t == 0].mean()
    return tpr, fpr

tpr0, fpr0 = group_rates(y, pred, group == 0)
tpr1, fpr1 = group_rates(y, pred, group == 1)

print(f"{'group':>8}{'TPR (approve true repayers)':>30}"
      f"{'FPR (approve true defaulters)':>32}")
print(f"{'0':>8}{tpr0:>30.4f}{fpr0:>32.4f}")
print(f"{'1':>8}{tpr1:>30.4f}{fpr1:>32.4f}")
print(f"\nTPR gap: {abs(tpr0-tpr1):.4f}    FPR gap: {abs(fpr0-fpr1):.4f}")

# calibration: among people the model scores at a given confidence
# level, does that confidence mean the same thing in both groups?
print(f"\n{'score bucket':>14}{'group 0 actual repay rate':>27}"
      f"{'group 1 actual repay rate':>27}")
for lo, hi in [(0.4, 0.5), (0.5, 0.6), (0.6, 0.7), (0.7, 0.8)]:
    mask = (p_repay >= lo) & (p_repay < hi)
    r0 = (y[(group == 0) & mask].mean()
          if (mask & (group == 0)).sum() > 0 else float('nan'))
    r1 = (y[(group == 1) & mask].mean()
          if (mask & (group == 1)).sum() > 0 else float('nan'))
    print(f"{f'{lo}-{hi}':>14}{r0:>27.4f}{r1:>27.4f}")

   group   TPR (approve true repayers)   FPR (approve true defaulters)
       0                        0.9841                          0.2101
       1                        0.8938                          0.0561

TPR gap: 0.0903    FPR gap: 0.1540

  score bucket  group 0 actual repay rate  group 1 actual repay rate
       0.4-0.5                     0.1875                     0.5682
       0.5-0.6                     0.3548                     0.7000
       0.6-0.7                     0.3548                     0.8254
       0.7-0.8                     0.6250                     0.8906


### Block 3  (`c3.py`)

In [5]:
# A common fix: use a different approval threshold per group so that
# false-positive rates match exactly. This is a real, deployable
# intervention. What it costs is consistency: the same predicted score
# now leads to a different decision depending on which group a person is in.
def fpr_at_threshold(p, y_true, group_mask, threshold):
    pred_t = (p >= threshold).astype(int)
    y_t, y_p = y_true[group_mask], pred_t[group_mask]
    return y_p[y_t == 0].mean()

# match group 1's original FPR
target_fpr = fpr_at_threshold(p_repay, y, group == 1, 0.5)

best_thresh, best_gap = 0.5, 1.0
for t in np.linspace(0.01, 0.99, 400):
    f = fpr_at_threshold(p_repay, y, group == 0, t)
    gap = abs(f - target_fpr)
    if gap < best_gap:
        best_gap, best_thresh = gap, t

pred_equalized = pred.copy()
pred_equalized[group == 0] = (p_repay[group == 0] >= best_thresh).astype(int)

tpr0_eq, fpr0_eq = group_rates(y, pred_equalized, group == 0)
tpr1_eq, fpr1_eq = group_rates(y, pred_equalized, group == 1)

print(f"to match group 1's false-positive rate of "
      f"{target_fpr:.4f}, group 0's")
print(f"approval threshold must move from 0.500 to {best_thresh:.3f}")
print(f"\n{'group':>8}{'threshold':>12}{'FPR':>10}{'TPR':>10}"
      f"{'approval rate':>16}")
print(f"{'0':>8}{best_thresh:>12.3f}{fpr0_eq:>10.4f}{tpr0_eq:>10.4f}"
      f"{pred_equalized[group==0].mean():>16.4f}")
print(f"{'1':>8}{0.5:>12.3f}{fpr1_eq:>10.4f}{tpr1_eq:>10.4f}"
      f"{pred_equalized[group==1].mean():>16.4f}")

print(f"\nFPR now matches almost exactly. But a person in group 0 with a")
print(f"predicted repayment probability of {best_thresh:.2f} "
      f"is now REJECTED,")
print(f"while a person in group 1 with the identical predicted probability")
print(f"of {best_thresh:.2f} is APPROVED, at the unchanged "
      f"threshold of 0.5.")
print(f"the same score now means two different decisions.")

to match group 1's false-positive rate of 0.0561, group 0's
approval threshold must move from 0.500 to 0.914

   group   threshold       FPR       TPR   approval rate
       0       0.914    0.0545    0.9048          0.6951
       1       0.500    0.0561    0.8938          0.5193

FPR now matches almost exactly. But a person in group 0 with a
predicted repayment probability of 0.91 is now REJECTED,
while a person in group 1 with the identical predicted probability
of 0.91 is APPROVED, at the unchanged threshold of 0.5.
the same score now means two different decisions.
